# Week 2, Lab 3 — Handoffs   Triage routes to a math specialist or a facts specialist.


In [ ]:
cfg = openai_client_kwargs()
print(cfg)

from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, function_tool, handoff

client = AsyncOpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"])
model = OpenAIChatCompletionsModel(model=cfg["model"], openai_client=client)


In [ ]:
from agents import function_tool, handoff

@function_tool
def calculator_tool(expression: str) -> str:
    """Evaluate arithmetic."""
    return calculator(expression)

@function_tool
def lookup_fact_tool(topic: str) -> str:
    """Local course facts."""
    return lookup_fact(topic)

math_agent = Agent(
    name="MathSpecialist",
    instructions="You only solve math. Always use calculator_tool.",
    model=model,
    tools=[calculator_tool],
)
facts_agent = Agent(
    name="FactsSpecialist",
    instructions="You only answer with lookup_fact_tool.",
    model=model,
    tools=[lookup_fact_tool],
)
triage = Agent(
    name="Triage",
    instructions="If the user asks math, hand off to MathSpecialist. If they ask about AI/agents topics, hand off to FactsSpecialist. Otherwise answer briefly yourself.",
    model=model,
    handoffs=[handoff(math_agent), handoff(facts_agent)],
)

for q in ["What is 19*21?", "What is Ollama?", "Say hello."]:
    result = await Runner.run(triage, q)
    print("Q:", q)
    print("A:", result.final_output)
    print("---")


## Exercise\n\nAdd a DateAgent specialist.\n\n**Next:** guardrails.
